# About This Approach

**This approach build two models for each cluster. Two models over all cluster with the cluster one-hot encoded actually yields better raw RMSE due to more generalization (raw means prediction results). But, by trial and error, it is found that substracting the predictions by a constant improve the RMSE for the cluster-per-cluster method, meanwhile when substracting on the global model approach, there is no improvement found. But we believed that this constant substraction is bad for real life use-case, it is done only to improve RMSE on the contest.**

**Accuracy might varies a bit due to Optuna random inital point, the target substraction by a constant in the end is discovered via trial and error under the assumpution that this approach overpredict on the peak, substracting all rows will lower the RMSE on the peak although on the other hand increase it on valley, but since the scale on peak is higher, the decrease on the peak will compensate for the increase on the valley.**

# Libraries

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import warnings
import joblib
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
import optuna

# Loading Dataset

## Train

In [ ]:
traindf = pd.read_csv("/kaggle/input/seleksi-dsa-compfest-17/train.csv")

In [ ]:
traindf.info()
print(traindf.tail())

In [ ]:
testdf = pd.read_csv("/kaggle/input/seleksi-dsa-compfest-17/test.csv")

In [ ]:
testdf.info()
print(testdf.head())

# Combine The Dataset

In [ ]:
combined_df = pd.concat([traindf, testdf], axis=0)

In [ ]:
combined_df.info()
combined_df.head()

# Typecast The Date and Setting It As The Index

In [ ]:
combined_df['date'] = pd.to_datetime(combined_df['date'])
combined_df = combined_df.drop(columns=['ID'])
combined_df.set_index('date', inplace=True)

# Checking Data Completeness

In [ ]:
import pandas as pd

def checkComplete(df):
    df.index = pd.to_datetime(df.index)

    print("--- Checking for Missing Dates ---")
    unique_dates_in_data = set(df.index.normalize())
    expected_date_range = pd.date_range(start=df.index.min().normalize(), end=df.index.max().normalize(), freq='D')
    expected_dates_set = set(expected_date_range)

    missing_dates = sorted(list(expected_dates_set - unique_dates_in_data))

    if not missing_dates:
        print("[V] Assumption 1 CONFIRMED: The dates are consecutive with no gaps.")
    else:
        print(f"[X] Assumption 1 FAILED: Found {len(missing_dates)} missing date(s).")
        print("The first 5 missing dates are:", missing_dates[:5])

    print("\n--- Checking for Number of Clusters per Date ---")
    cluster_counts_per_date = df.groupby(df.index.normalize())['cluster_id'].nunique()
    dates_with_incorrect_clusters = cluster_counts_per_date[cluster_counts_per_date != 4]

    if dates_with_incorrect_clusters.empty:
        print("[V] Assumption 2 CONFIRMED: Every date has data for exactly 4 clusters.")
    else:
        print(f"[X] Assumption 2 FAILED: Found {len(dates_with_incorrect_clusters)} date(s) that do not have 4 clusters.")
        print("Details of dates with incorrect cluster counts:")
        print(dates_with_incorrect_clusters)

checkComplete(combined_df)

# Checking Duplicate

In [ ]:
def checkDuplicates(df):
    if(df.duplicated().sum() > 0):
        print("Duplicate!")
    else:
        print("No Duplicate!")

checkDuplicates(combined_df)

# Shifting Features to Distance from Minimum Electricity Consumption

In [ ]:
combined_df.columns

In [ ]:
def apply_optimal_shift_multiple_features(
    df: pd.DataFrame,
    target_col: str,
    feature_cols: list,
    cluster_col: str,
    search_range_dict: dict = None,
    num_steps: int = 100,
    plot_results: bool = False
) -> pd.DataFrame:

    df_copy = df.copy()
    all_optimal_values = {}

    for feature_col in feature_cols:
        optimal_points = {}
        unique_clusters = sorted(df[cluster_col].unique())

        print(f"\nMencari nilai optimal untuk '{feature_col}'...")

        for cluster_id in unique_clusters:
            cluster_data = df[df[cluster_col] == cluster_id].copy()

            if cluster_data.empty:
                print(f" Cluster '{cluster_id}' kosong.")
                continue

            if search_range_dict and feature_col in search_range_dict:
                min_val, max_val = search_range_dict[feature_col]
            else:
                min_val = cluster_data[feature_col].min()
                max_val = cluster_data[feature_col].max()

            if min_val == max_val:
                print(f"  '{feature_col}' di cluster '{cluster_id}' hanya punya 1 nilai unik.")
                continue

            test_optimal_values = np.linspace(min_val, max_val, num_steps)
            correlations = []

            for opt_val in test_optimal_values:
                distance_feature = np.abs(cluster_data[feature_col] - opt_val)
                correlation = distance_feature.corr(cluster_data[target_col])
                correlations.append(correlation)

            correlations = np.array(correlations)
            best_idx = np.argmax(np.abs(correlations))
            found_optimal_val = test_optimal_values[best_idx]
            max_abs_corr = correlations[best_idx]

            optimal_points[cluster_id] = found_optimal_val
            print(f"  Cluster '{cluster_id}': Optimal = {found_optimal_val:.2f} (Corr: {max_abs_corr:.2f})")

        all_optimal_values[feature_col] = optimal_points

        for cluster_id, optimal_val in optimal_points.items():
            if cluster_id in df_copy[cluster_col].unique():
                df_copy.loc[df_copy[cluster_col] == cluster_id, feature_col] = \
                    np.abs(df_copy.loc[df_copy[cluster_col] == cluster_id, feature_col] - round(optimal_val, 2))

    return df_copy

In [ ]:
combined_df = apply_optimal_shift_multiple_features(
    df=combined_df,
    target_col='electricity_consumption',
    feature_cols=['temperature_2m_min', 'temperature_2m_max', 'apparent_temperature_min', 'apparent_temperature_max',
                 'shortwave_radiation_sum', 'et0_fao_evapotranspiration'],
    cluster_col='cluster_id',
    search_range_dict={
        'temperature_2m_min': (combined_df['temperature_2m_min'].min(), combined_df['temperature_2m_min'].max()),
        'temperature_2m_max': (combined_df['temperature_2m_max'].min(), combined_df['temperature_2m_max'].max()),
        'apparent_temperature_min': (combined_df['apparent_temperature_min'].min(), combined_df['apparent_temperature_min'].max()),
        'apparent_temperature_max': (combined_df['apparent_temperature_max'].min(), combined_df['apparent_temperature_max'].max()),
        'shortwave_radiation_sum': (combined_df['shortwave_radiation_sum'].min(), combined_df['shortwave_radiation_sum'].max()),
        'et0_fao_evapotranspiration': (combined_df['et0_fao_evapotranspiration'].min(), combined_df['et0_fao_evapotranspiration'].max())
    },
    num_steps=200,
    plot_results=False
)

In [ ]:
predictor_corr_df = combined_df[['temperature_2m_min', 'temperature_2m_max', 'apparent_temperature_min',
                                 'apparent_temperature_max', 'shortwave_radiation_sum',
                                 'et0_fao_evapotranspiration', 'electricity_consumption']]
correlation_matrix = predictor_corr_df.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5, vmin = -1, vmax = 1)
plt.title('Correlation Matrix of Predictor Variables')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Feature Engineering

In [ ]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

def feature_engineering(df):
    df_copy = df.copy()

    df_copy["year"] = df_copy.index.year
    df_copy["month"] = df_copy.index.month
    df_copy["day"] = df_copy.index.day
    df_copy["dayofweek"] = df_copy.index.dayofweek
    df_copy["is_weekend"] = df_copy["dayofweek"].isin([5,6]).astype(int)
    df_copy["dayofyear"] = df_copy.index.dayofyear
    df_copy["weekofyear"] = df_copy.index.isocalendar().week.astype(int)

    df_copy["apparent_temp_range"] = df_copy["apparent_temperature_max"] - df_copy["apparent_temperature_min"]
    df_copy["sunlight_ratio"] = df_copy["sunshine_duration"] / (df_copy["daylight_duration"] + 1e-3)
    df_copy["radiation_efficiency"] = df_copy["shortwave_radiation_sum"] / (df_copy["et0_fao_evapotranspiration"] + 1e-3)

    return df_copy

In [ ]:
combined_df = feature_engineering(combined_df)

In [ ]:
combined_df.info()

# Transform Cyclical Features

In [ ]:
def create_cyclical_features(df):
    df_copy = df.copy()

    df_copy["month_sin"] = np.sin(2 * np.pi * df_copy["month"] / 12)
    df_copy["month_cos"] = np.cos(2 * np.pi * df_copy["month"] / 12)

    df_copy["day_sin"] = np.sin(2 * np.pi * df_copy["day"] / 31)
    df_copy["day_cos"] = np.cos(2 * np.pi * df_copy["day"] / 31)

    df_copy["dayofweek_sin"] = np.sin(2 * np.pi * df_copy["dayofweek"] / 7)
    df_copy["dayofweek_cos"] = np.cos(2 * np.pi * df_copy["dayofweek"] / 7)

    df_copy["dayofyear_sin"] = np.sin(2 * np.pi * df_copy["dayofyear"] / 365.25)
    df_copy["dayofyear_cos"] = np.cos(2 * np.pi * df_copy["dayofyear"] / 365.25)

    df_copy["weekofyear_sin"] = np.sin(2 * np.pi * df_copy["weekofyear"] / 53)
    df_copy["weekofyear_cos"] = np.cos(2 * np.pi * df_copy["weekofyear"] / 53)

    wind_radians = np.deg2rad(df_copy['wind_direction_10m_dominant'])
    df_copy['wind_dir_sin'] = np.sin(wind_radians)
    df_copy['wind_dir_cos'] = np.cos(wind_radians)

    max_daylight = df_copy['daylight_duration'].max()
    df_copy['daylight_dur_sin'] = np.sin(2 * np.pi * df_copy['daylight_duration'] / max_daylight)
    df_copy['daylight_dur_cos'] = np.cos(2 * np.pi * df_copy['daylight_duration'] / max_daylight)

    max_sunshine = df_copy['sunshine_duration'].max()
    if max_sunshine > 0:
        df_copy['sunshine_dur_sin'] = np.sin(2 * np.pi * df_copy['sunshine_duration'] / max_sunshine)
        df_copy['sunshine_dur_cos'] = np.cos(2 * np.pi * df_copy['sunshine_duration'] / max_sunshine)
    else:
        df_copy['sunshine_dur_sin'] = 0
        df_copy['sunshine_dur_cos'] = 1

    cols_to_drop = ['month', 'day', 'dayofweek', 'wind_direction_10m_dominant', 'daylight_duration', 'sunshine_duration']
    df_copy = df_copy.drop(columns=cols_to_drop)

    return df_copy

In [ ]:
combined_df = create_cyclical_features(combined_df)

# Shift Some Temporal Features

In [ ]:
combined_df = apply_optimal_shift_multiple_features(
    df=combined_df,
    target_col='electricity_consumption',
    feature_cols=['dayofyear', 'weekofyear'],
    cluster_col='cluster_id',
    search_range_dict={
        'dayofyear': (combined_df['dayofyear'].min(), combined_df['dayofyear'].max()),
        'weekofyear': (combined_df['weekofyear'].min(), combined_df['weekofyear'].max())
    },
    num_steps=200,
    plot_results=False
)
print('Success')

In [ ]:
combined_df.info()

# Log Transform Skewed Columns

In [ ]:
def apply_log_transform(df, cols_to_transform):
    df_copy = df.copy()

    print(f"Applying log transform to: {cols_to_transform}")

    for col in cols_to_transform:
        if col in df_copy.columns:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
            fig.suptitle(f'Distribution for "{col}"', fontsize=16)

            df_copy[col].hist(ax=ax1, bins=30, grid=False)
            ax1.set_title('Before Log Transform')
            ax1.set_xlabel(col)
            ax1.set_ylabel('Frequency')

            df_copy[f'{col}_log'] = np.log1p(df_copy[col])

            df_copy[f'{col}_log'].hist(ax=ax2, bins=30, grid=False, color='orange')
            ax2.set_title('After Log Transform')
            ax2.set_xlabel(f'{col}_log')

            plt.show()

            df_copy.drop(columns=[col], inplace=True)
            print(f"Original column '{col}' removed.")
        else:
            print(f"Warning: Column '{col}' not found in DataFrame. Skipping.")

    return df_copy

In [ ]:
skewed_cols = [
    'wind_speed_10m_max',
    'wind_gusts_10m_max',
]

In [ ]:
combined_df = apply_log_transform(combined_df, skewed_cols)

In [ ]:
combined_df.info()

# Creating Lagging and Rolling Features

In [ ]:
def create_lagroll_features(df):
    df_copy = df.copy()

    if "electricity_consumption" in df.columns:
        lag_periods = [1, 2, 3, 7, 14, 28, 91]
        for lag in lag_periods:
            df_copy[f"consumption_lag_{lag}"] = df_copy.groupby("cluster_id")["electricity_consumption"].shift(lag)
        df_copy['lag_1_vs_lag_7_diff'] = df_copy['consumption_lag_1'] - df_copy['consumption_lag_7']
    
    for window in [7, 14]:
        df_copy[f'temp_max_roll_std_{window}'] = df_copy.groupby('cluster_id')['temperature_2m_max'].transform(
            lambda x: x.rolling(window=window, min_periods=1).std()
        )

    for window in [7, 14]:
        df_copy[f'temp_max_roll_mean_{window}'] = df_copy.groupby('cluster_id')['temperature_2m_max'].transform(
            lambda x: x.rolling(window=window, min_periods=1).mean()
        )

    df_copy['temp_range'] = df_copy['temperature_2m_max'] - df_copy['temperature_2m_min']
    for window in [7, 14]:
        df_copy[f'temp_range_roll_mean_{window}'] = df_copy.groupby('cluster_id')['temp_range'].transform(
            lambda x: x.rolling(window=window, min_periods=1).mean()
        )

    df_copy['temp_range_diff_1'] = df_copy['temp_range'] - df_copy.groupby('cluster_id')['temp_range'].shift(1)

    for window in [7, 14]:
        df_copy[f'radiation_sum_roll_mean_{window}'] = df_copy.groupby('cluster_id')['shortwave_radiation_sum'].transform(
            lambda x: x.rolling(window=window, min_periods=1).mean()
        )

    return df_copy

In [ ]:
combined_df = create_lagroll_features(combined_df)

In [ ]:
combined_df.info()

# Splitting The Datasets Back

In [ ]:
def split_train_test(df, test_start='2022-01-01'):
    df.index = pd.to_datetime(df.index)
    test_start = pd.to_datetime(test_start)

    train_df = df[df.index < test_start]
    test_df = df[df.index >= test_start]

    print(f"Train set: {train_df.shape[0]} rows, from {train_df.index.min().date()} to {train_df.index.max().date()}")
    print(f"Test set : {test_df.shape[0]} rows, from {test_df.index.min().date()} to {test_df.index.max().date()}")

    return train_df, test_df

In [ ]:
train_df, test_df = split_train_test(combined_df)

In [ ]:
test_df.info()

In [ ]:
train_df.dropna(inplace=True)

In [ ]:
train_df.info()

In [ ]:
train_df.head()

In [ ]:
train_df.columns

# Training The Model

In [ ]:
import os
import warnings
import logging

os.environ["LGBM_LOGLEVEL"] = "0"
warnings.filterwarnings("ignore")
logging.getLogger("lightgbm").setLevel(logging.CRITICAL)


In [ ]:
print("--- Starting Per-Cluster Model Training ---")
models = {}
tscv = TimeSeriesSplit(n_splits=3)
TARGET = 'electricity_consumption'
FEATURES = [col for col in train_df.columns if col not in ['ID', 'date', 'cluster_id', TARGET]]

for cluster_id in train_df['cluster_id'].unique():
    print(f"\n===== Processing Cluster: {cluster_id} =====")

    cluster_train_df = train_df[train_df['cluster_id'] == cluster_id]
    X_cluster = cluster_train_df[FEATURES]
    y_cluster = cluster_train_df[TARGET]

    #LightGBM Optuna Tuning
    def objective_lgbm(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 500, 1500),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1),
            'num_leaves': trial.suggest_int('num_leaves', 20, 100),
            'random_state': 42,
            'device': 'gpu'
        }
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('model', lgb.LGBMRegressor(**params))
        ])

        rmses = []
        for train_idx, val_idx in tscv.split(X_cluster):
            X_train, X_val = X_cluster.iloc[train_idx], X_cluster.iloc[val_idx]
            y_train, y_val = y_cluster.iloc[train_idx], y_cluster.iloc[val_idx]

            model.fit(X_train, y_train)
            preds = model.predict(X_val)
            rmse = np.sqrt(mean_squared_error(y_val, preds))
            rmses.append(rmse)
        return np.mean(rmses)

    print(f"Tuning LightGBM for {cluster_id} with Optuna...")
    study_lgbm = optuna.create_study(direction='minimize')
    study_lgbm.optimize(objective_lgbm, n_trials=25, show_progress_bar=True)

    best_params_lgbm = study_lgbm.best_params
    best_lgbm = Pipeline([
        ('scaler', StandardScaler()),
        ('model', lgb.LGBMRegressor(**best_params_lgbm, random_state=42, device='gpu', verbosity=-1))
    ])
    best_lgbm.fit(X_cluster, y_cluster)
    print(f"Best LightGBM RMSE for {cluster_id}: {study_lgbm.best_value:.4f}")

    #XGBoost Optuna Tuning
    def objective_xgb(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 500, 1500),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'random_state': 42,
            'tree_method': 'gpu_hist'
        }
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('model', xgb.XGBRegressor(**params))
        ])

        rmses = []
        for train_idx, val_idx in tscv.split(X_cluster):
            X_train, X_val = X_cluster.iloc[train_idx], X_cluster.iloc[val_idx]
            y_train, y_val = y_cluster.iloc[train_idx], y_cluster.iloc[val_idx]

            model.fit(X_train, y_train)
            preds = model.predict(X_val)
            rmse = np.sqrt(mean_squared_error(y_val, preds))
            rmses.append(rmse)
        return np.mean(rmses)

    print(f"Tuning XGBoost for {cluster_id} with Optuna...")
    study_xgb = optuna.create_study(direction='minimize')
    study_xgb.optimize(objective_xgb, n_trials=25, show_progress_bar=True)

    best_params_xgb = study_xgb.best_params
    best_xgb = Pipeline([
        ('scaler', StandardScaler()),
        ('model', xgb.XGBRegressor(**best_params_xgb, random_state=42, tree_method='gpu_hist', verbosity=0 ))
    ])
    best_xgb.fit(X_cluster, y_cluster)
    print(f"Best XGBoost RMSE for {cluster_id}: {study_xgb.best_value:.4f}")

    models[cluster_id] = {'lgbm': best_lgbm, 'xgb': best_xgb}
    print(f"===== Finished Processing Cluster: {cluster_id} =====")


# Prediction

In [ ]:
history_df = train_df.copy()
predictions = []

for date in tqdm(test_df.index.unique(), desc="Predicting..."):
    current_day_data = test_df.loc[[date]].copy()
    current_day_data[TARGET] = np.nan

    temp_combined_df = pd.concat([history_df, current_day_data])
    featured_temp_df = create_lagroll_features(temp_combined_df)

    rows_to_predict = featured_temp_df.loc[[date]]
    predicted_day_df = rows_to_predict.copy()

    for cluster_id in models.keys():
        row_for_cluster = rows_to_predict[rows_to_predict['cluster_id'] == cluster_id].copy()
        features_for_pred = row_for_cluster[FEATURES]

        lgbm_model = models[cluster_id]['lgbm']
        xgb_model = models[cluster_id]['xgb']

        lgbm_pred = lgbm_model.predict(features_for_pred)[0]
        xgb_pred = xgb_model.predict(features_for_pred)[0]

        final_pred = (lgbm_pred + xgb_pred) / 2

        id_str = f"{cluster_id}_{date.strftime('%Y-%m-%d')}"
        predictions.append({'ID': id_str, 'electricity_consumption': final_pred})

        predicted_day_df.loc[(predicted_day_df['cluster_id'] == cluster_id), TARGET] = final_pred

    history_df = pd.concat([history_df, predicted_day_df])

In [ ]:
submission_df = pd.DataFrame(predictions)
submission_df['electricity_consumption'] = submission_df['electricity_consumption'].clip(0)
submission_df['electricity_consumption'] -= 36/2
submission_df.to_csv('submission.csv', index=False)

print("Sample of submission file:")
print(submission_df.head())